# 03 Feature Engineering — Scope and Leakage Rules

This notebook creates deterministic, leakage-free feature matrices for the MoA prediction project.

The goal is to build feature-set versions that can later be compared during model training:

- FE_V0: raw metadata + gene + cell features
- FE_V1: FE_V0 + row-wise gene/cell summary features
- FE_V2: FE_V1 + small gene-cell interaction features
- FE_V3: optional metadata interaction features

This notebook will not train models, fit scalers, fit PCA, fit quantile transformers, fit feature selectors, or use any target-derived information as input features.

All learned transformations such as scaling, PCA, quantile transformation, variance filtering, and supervised feature selection must be fitted only inside the validation/modeling pipeline.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Project root detection
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

EXPERIMENT_LOG_DIR = PROJECT_ROOT / "outputs" / "experiment_logs"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_LOG_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW_DIR:", DATA_RAW_DIR)
print("DATA_INTERIM_DIR:", DATA_INTERIM_DIR)
print("DATA_PROCESSED_DIR:", DATA_PROCESSED_DIR)
print("EXPERIMENT_LOG_DIR:", EXPERIMENT_LOG_DIR)
print("FIGURE_DIR:", FIGURE_DIR)

PROJECT_ROOT: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response
DATA_RAW_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\raw
DATA_INTERIM_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\interim
DATA_PROCESSED_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\processed
EXPERIMENT_LOG_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\outputs\experiment_logs
FIGURE_DIR: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\reports\figures


In [3]:
feature_engineering_scope_table = pd.DataFrame({
    "area": [
        "Notebook purpose",
        "Allowed feature type",
        "Baseline feature versions",
        "Model training",
        "Learned transformations",
        "Target-derived features",
        "Nonscored targets",
        "Drug ID",
        "Control-row prediction rule"
    ],
    "rule": [
        "Create leakage-free deterministic feature matrices",
        "Only input-derived, row-wise features",
        "FE_V0, FE_V1, FE_V2, optional FE_V3",
        "Not performed in this notebook",
        "Not fitted in this notebook",
        "Never used as model input",
        "Never used as baseline input features",
        "Not used as baseline model input",
        "Documented here, applied only after prediction"
    ],
    "reason": [
        "Feature engineering should stay separate from modeling",
        "Safe because each feature uses only the same sample's input values",
        "Allows clean model comparison later",
        "Model training belongs to model training notebook",
        "Scaler/PCA/Quantile/selection must be fitted inside CV pipeline",
        "Would leak label information",
        "Nonscored labels are auxiliary targets only, not inputs",
        "Drug ID is useful for validation diagnostics but risky as baseline input",
        "ctl_vehicle zeroing is an inference/submission post-processing rule"
    ]
})

feature_engineering_scope_table

,area,rule,reason
0,Notebook purpose,Create leakage-free deterministic feature matr...,Feature engineering should stay separate from ...
1,Allowed feature type,"Only input-derived, row-wise features",Safe because each feature uses only the same s...
2,Baseline feature versions,"FE_V0, FE_V1, FE_V2, optional FE_V3",Allows clean model comparison later
3,Model training,Not performed in this notebook,Model training belongs to model training notebook
4,Learned transformations,Not fitted in this notebook,Scaler/PCA/Quantile/selection must be fitted i...
5,Target-derived features,Never used as model input,Would leak label information
6,Nonscored targets,Never used as baseline input features,"Nonscored labels are auxiliary targets only, n..."
7,Drug ID,Not used as baseline model input,Drug ID is useful for validation diagnostics b...
8,Control-row prediction rule,"Documented here, applied only after prediction",ctl_vehicle zeroing is an inference/submission...


In [4]:
hard_leakage_rules = pd.DataFrame({
    "rule_id": [
        "L001",
        "L002",
        "L003",
        "L004",
        "L005",
        "L006",
        "L007",
        "L008",
        "L009",
        "L010"
    ],
    "leakage_rule": [
        "No scored target column can enter X",
        "No nonscored target column can enter X",
        "No target-derived count or flag can enter X",
        "No fold-level target statistic can enter X",
        "No target co-occurrence feature can enter X",
        "No scaler is fitted before cross-validation",
        "No PCA is fitted before cross-validation",
        "No QuantileTransformer is fitted before cross-validation",
        "No feature selector is fitted before cross-validation",
        "No drug_id is used as a baseline input feature"
    ],
    "forbidden_examples": [
        "nfkb_inhibitor, proteasome_inhibitor, etc.",
        "any nonscored target label",
        "active_target_count, zero_label_flag, multi_label_count",
        "validation_positive_count, rare_label_fold_count",
        "target_pair_count, label_cluster_id",
        "StandardScaler.fit(full_train)",
        "PCA.fit(full_train)",
        "QuantileTransformer.fit(full_train)",
        "SelectKBest.fit(full_train, y)",
        "drug_id one-hot as baseline X"
    ],
    "safe_alternative": [
        "Use targets only as y",
        "Use nonscored only as advanced auxiliary y",
        "Use gene/cell input-derived summaries",
        "Use fold statistics only for diagnostics",
        "Use co-occurrence only for EDA interpretation",
        "Fit scaler inside modeling pipeline",
        "Fit PCA inside modeling pipeline",
        "Fit QuantileTransformer inside modeling pipeline",
        "Fit selector inside modeling pipeline",
        "Use drug_id only for validation-risk diagnostics"
    ]
})

hard_leakage_rules

,rule_id,leakage_rule,forbidden_examples,safe_alternative
0,L001,No scored target column can enter X,"nfkb_inhibitor, proteasome_inhibitor, etc.",Use targets only as y
1,L002,No nonscored target column can enter X,any nonscored target label,Use nonscored only as advanced auxiliary y
2,L003,No target-derived count or flag can enter X,"active_target_count, zero_label_flag, multi_la...",Use gene/cell input-derived summaries
3,L004,No fold-level target statistic can enter X,"validation_positive_count, rare_label_fold_count",Use fold statistics only for diagnostics
4,L005,No target co-occurrence feature can enter X,"target_pair_count, label_cluster_id",Use co-occurrence only for EDA interpretation
5,L006,No scaler is fitted before cross-validation,StandardScaler.fit(full_train),Fit scaler inside modeling pipeline
6,L007,No PCA is fitted before cross-validation,PCA.fit(full_train),Fit PCA inside modeling pipeline
7,L008,No QuantileTransformer is fitted before cross-...,QuantileTransformer.fit(full_train),Fit QuantileTransformer inside modeling pipeline
8,L009,No feature selector is fitted before cross-val...,"SelectKBest.fit(full_train, y)",Fit selector inside modeling pipeline
9,L010,No drug_id is used as a baseline input feature,drug_id one-hot as baseline X,Use drug_id only for validation-risk diagnostics


In [5]:
feature_category_rules = pd.DataFrame({
    "feature_category": [
        "Raw metadata",
        "Raw gene features",
        "Raw cell features",
        "Gene summary features",
        "Cell summary features",
        "Gene-cell interaction features",
        "Metadata interaction features",
        "Scaling",
        "PCA",
        "Quantile transformation",
        "Feature selection",
        "Nonscored targets",
        "Scored targets",
        "Drug ID",
        "Control-row zeroing"
    ],
    "status": [
        "allowed",
        "allowed",
        "allowed",
        "allowed",
        "allowed",
        "allowed",
        "optional",
        "pipeline-only",
        "pipeline-only",
        "pipeline-only",
        "pipeline-only",
        "forbidden-as-input",
        "forbidden-as-input",
        "diagnostic-only",
        "inference-only"
    ],
    "where_used": [
        "FE_V0 onward",
        "FE_V0 onward",
        "FE_V0 onward",
        "FE_V1 onward",
        "FE_V1 onward",
        "FE_V2 onward",
        "optional FE_V3",
        "modeling pipeline",
        "modeling pipeline",
        "modeling pipeline",
        "modeling pipeline",
        "advanced auxiliary output only",
        "y target only",
        "validation-risk analysis only",
        "submission post-processing"
    ],
    "notes": [
        "cp_type, cp_time, cp_dose",
        "g-* columns",
        "c-* columns",
        "row-wise deterministic",
        "row-wise deterministic",
        "small audited set only",
        "keep small; avoid sparse explosion",
        "fit on training folds only",
        "fit on training folds only",
        "fit on training folds only",
        "fit on training folds only",
        "never concatenate into X",
        "never concatenate into X",
        "not a baseline feature",
        "apply after prediction, not during feature creation"
    ]
})

feature_category_rules

,feature_category,status,where_used,notes
0,Raw metadata,allowed,FE_V0 onward,"cp_type, cp_time, cp_dose"
1,Raw gene features,allowed,FE_V0 onward,g-* columns
2,Raw cell features,allowed,FE_V0 onward,c-* columns
3,Gene summary features,allowed,FE_V1 onward,row-wise deterministic
4,Cell summary features,allowed,FE_V1 onward,row-wise deterministic
5,Gene-cell interaction features,allowed,FE_V2 onward,small audited set only
6,Metadata interaction features,optional,optional FE_V3,keep small; avoid sparse explosion
7,Scaling,pipeline-only,modeling pipeline,fit on training folds only
8,PCA,pipeline-only,modeling pipeline,fit on training folds only
9,Quantile transformation,pipeline-only,modeling pipeline,fit on training folds only


In [6]:
feature_engineering_scope_table.to_csv(
    EXPERIMENT_LOG_DIR / "step_1_feature_engineering_scope_table.csv",
    index=False
)

hard_leakage_rules.to_csv(
    EXPERIMENT_LOG_DIR / "step_1_hard_leakage_rules.csv",
    index=False
)

feature_category_rules.to_csv(
    EXPERIMENT_LOG_DIR / "step_1_feature_category_rules.csv",
    index=False
)

print("Step 1 scope and leakage-rule files saved successfully.")

Step 1 scope and leakage-rule files saved successfully.


In [7]:
step_1_status = {
    "step": "Step 1 — Notebook Scope and Leakage Rules",
    "status": "completed",
    "model_training_started": False,
    "learned_transform_fitted": False,
    "target_derived_features_allowed": False,
    "nonscored_targets_allowed_as_input": False,
    "drug_id_allowed_as_baseline_input": False,
    "ready_for_step_2": True,
    "next_step": "Step 2 — Load Clean Interim Data"
}

step_1_status

{'step': 'Step 1 — Notebook Scope and Leakage Rules',
 'status': 'completed',
 'model_training_started': False,
 'learned_transform_fitted': False,
 'target_derived_features_allowed': False,
 'nonscored_targets_allowed_as_input': False,
 'drug_id_allowed_as_baseline_input': False,
 'ready_for_step_2': True,
 'next_step': 'Step 2 — Load Clean Interim Data'}

## Step 2 — Load Clean Interim Data

This step loads the clean interim datasets created during data integration.

This notebook does not repeat raw data integration.  
The goal is to start feature engineering from already validated clean assets.

Loaded assets:

- clean train features
- clean test features
- scored targets
- nonscored targets
- train drug table
- feature group dictionary
- sample submission

In [8]:
expected_interim_files = {
    "train_features_clean": DATA_INTERIM_DIR / "train_features_clean.parquet",
    "test_features_clean": DATA_INTERIM_DIR / "test_features_clean.parquet",
    "y_scored": DATA_INTERIM_DIR / "y_scored.parquet",
    "y_nonscored": DATA_INTERIM_DIR / "y_nonscored.parquet",
    "train_drug_clean": DATA_INTERIM_DIR / "train_drug_clean.parquet",
    "feature_groups": DATA_INTERIM_DIR / "feature_groups.json"
}

expected_raw_files = {
    "sample_submission": DATA_RAW_DIR / "sample_submission.csv"
}

file_availability_audit = []

for file_name, file_path in {**expected_interim_files, **expected_raw_files}.items():
    file_availability_audit.append({
        "file_name": file_name,
        "expected_path": str(file_path),
        "exists": file_path.exists()
    })

file_availability_audit = pd.DataFrame(file_availability_audit)

file_availability_audit

,file_name,expected_path,exists
0,train_features_clean,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
1,test_features_clean,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
2,y_scored,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
3,y_nonscored,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
4,train_drug_clean,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
5,feature_groups,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
6,sample_submission,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True


In [9]:
missing_files = file_availability_audit.loc[
    file_availability_audit["exists"] == False,
    ["file_name", "expected_path"]
]

if len(missing_files) > 0:
    display(missing_files)
    raise FileNotFoundError("Some required clean/interim files are missing. Check paths before continuing.")

print("All required clean/interim files are available.")

All required clean/interim files are available.


In [11]:
def read_table_with_repair(parquet_path, raw_csv_path=None, dataset_name="dataset"):
    """
    Try to read parquet safely.
    If parquet fails, optionally reload from raw CSV and rewrite parquet.
    """
    parquet_path = Path(parquet_path)
    raw_csv_path = Path(raw_csv_path) if raw_csv_path is not None else None
    
    try:
        df = pd.read_parquet(parquet_path, engine="pyarrow")
        print(f"{dataset_name}: loaded from parquet with pyarrow")
        return df
    
    except Exception as pyarrow_error:
        print(f"{dataset_name}: pyarrow parquet read failed.")
        print("Error:", pyarrow_error)
        
        try:
            df = pd.read_parquet(parquet_path, engine="fastparquet")
            print(f"{dataset_name}: loaded from parquet with fastparquet")
            return df
        
        except Exception as fastparquet_error:
            print(f"{dataset_name}: fastparquet parquet read also failed.")
            print("Error:", fastparquet_error)
            
            if raw_csv_path is not None and raw_csv_path.exists():
                print(f"{dataset_name}: loading from raw CSV and repairing parquet...")
                df = pd.read_csv(raw_csv_path)
                
                df.to_parquet(
                    parquet_path,
                    index=False,
                    engine="pyarrow"
                )
                
                print(f"{dataset_name}: repaired parquet saved to {parquet_path}")
                return df
            
            raise RuntimeError(
                f"{dataset_name}: Could not read parquet and no valid raw CSV fallback found."
            )


raw_csv_fallback_files = {
    "train_features_clean": DATA_RAW_DIR / "train_features.csv",
    "test_features_clean": DATA_RAW_DIR / "test_features.csv",
    "y_scored": DATA_RAW_DIR / "train_targets_scored.csv",
    "y_nonscored": DATA_RAW_DIR / "train_targets_nonscored.csv",
    "train_drug_clean": DATA_RAW_DIR / "train_drug.csv"
}


train_features_clean = read_table_with_repair(
    expected_interim_files["train_features_clean"],
    raw_csv_fallback_files["train_features_clean"],
    "train_features_clean"
)

test_features_clean = read_table_with_repair(
    expected_interim_files["test_features_clean"],
    raw_csv_fallback_files["test_features_clean"],
    "test_features_clean"
)

y_scored = read_table_with_repair(
    expected_interim_files["y_scored"],
    raw_csv_fallback_files["y_scored"],
    "y_scored"
)

y_nonscored = read_table_with_repair(
    expected_interim_files["y_nonscored"],
    raw_csv_fallback_files["y_nonscored"],
    "y_nonscored"
)

train_drug_clean = read_table_with_repair(
    expected_interim_files["train_drug_clean"],
    raw_csv_fallback_files["train_drug_clean"],
    "train_drug_clean"
)

sample_submission = pd.read_csv(expected_raw_files["sample_submission"])

with open(expected_interim_files["feature_groups"], "r", encoding="utf-8") as f:
    feature_groups = json.load(f)

print("Clean data loaded successfully.")

train_features_clean: pyarrow parquet read failed.
Error: Repetition level histogram size mismatch
train_features_clean: fastparquet parquet read also failed.
Error: Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
train_features_clean: loading from raw CSV and repairing parquet...
train_features_clean: repaired parquet saved to C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\moa-prediction-drug-response\data\interim\train_features_clean.parquet
test_features_clean: pyarrow parquet read failed.
Error: Repetition level histogram size mismatch
test_features_clean: fastparquet parquet read also failed.
Error: Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
test_features_clean: loading from raw CSV and repairing parquet...
test_features_clean: repaired parquet saved to C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Mach

In [13]:
data_shape_audit = pd.DataFrame({
    "dataset": [
        "train_features_clean",
        "test_features_clean",
        "y_scored",
        "y_nonscored",
        "train_drug_clean",
        "sample_submission"
    ],
    "rows": [
        train_features_clean.shape[0],
        test_features_clean.shape[0],
        y_scored.shape[0],
        y_nonscored.shape[0],
        train_drug_clean.shape[0],
        sample_submission.shape[0]
    ],
    "columns": [
        train_features_clean.shape[1],
        test_features_clean.shape[1],
        y_scored.shape[1],
        y_nonscored.shape[1],
        train_drug_clean.shape[1],
        sample_submission.shape[1]
    ]
})

data_shape_audit

,dataset,rows,columns
0,train_features_clean,23814,876
1,test_features_clean,3982,876
2,y_scored,23814,207
3,y_nonscored,23814,403
4,train_drug_clean,23814,2
5,sample_submission,3982,207


In [14]:
def get_feature_group(feature_groups_dict, possible_keys, default=None):
    for key in possible_keys:
        if key in feature_groups_dict:
            return feature_groups_dict[key]
    return default if default is not None else []


ID_COL = "sig_id"

metadata_features = get_feature_group(
    feature_groups,
    ["metadata_features", "metadata", "meta_features"],
    default=["cp_type", "cp_time", "cp_dose"]
)

gene_features = get_feature_group(
    feature_groups,
    ["gene_features", "gene", "genes"],
    default=[col for col in train_features_clean.columns if col.startswith("g-")]
)

cell_features = get_feature_group(
    feature_groups,
    ["cell_features", "cell", "cells"],
    default=[col for col in train_features_clean.columns if col.startswith("c-")]
)

scored_target_features = get_feature_group(
    feature_groups,
    ["scored_target_features", "scored_targets", "target_features", "targets_scored"],
    default=[col for col in y_scored.columns if col != ID_COL]
)

nonscored_target_features = get_feature_group(
    feature_groups,
    ["nonscored_target_features", "nonscored_targets", "target_features_nonscored", "targets_nonscored"],
    default=[col for col in y_nonscored.columns if col != ID_COL]
)

feature_group_count_audit = pd.DataFrame({
    "feature_group": [
        "metadata_features",
        "gene_features",
        "cell_features",
        "scored_target_features",
        "nonscored_target_features"
    ],
    "count": [
        len(metadata_features),
        len(gene_features),
        len(cell_features),
        len(scored_target_features),
        len(nonscored_target_features)
    ]
})

feature_group_count_audit

,feature_group,count
0,metadata_features,3
1,gene_features,772
2,cell_features,100
3,scored_target_features,206
4,nonscored_target_features,402


In [15]:
id_alignment_audit = {
    "train_sig_id_unique": train_features_clean[ID_COL].is_unique,
    "test_sig_id_unique": test_features_clean[ID_COL].is_unique,
    "y_scored_sig_id_unique": y_scored[ID_COL].is_unique,
    "y_nonscored_sig_id_unique": y_nonscored[ID_COL].is_unique,
    "train_drug_sig_id_unique": train_drug_clean[ID_COL].is_unique,
    "sample_submission_sig_id_unique": sample_submission[ID_COL].is_unique,
    
    "train_vs_y_scored_order_match": train_features_clean[ID_COL].tolist() == y_scored[ID_COL].tolist(),
    "train_vs_y_nonscored_order_match": train_features_clean[ID_COL].tolist() == y_nonscored[ID_COL].tolist(),
    "train_vs_drug_order_match": train_features_clean[ID_COL].tolist() == train_drug_clean[ID_COL].tolist(),
    "test_vs_submission_order_match": test_features_clean[ID_COL].tolist() == sample_submission[ID_COL].tolist(),
    
    "train_rows_match_y_scored": train_features_clean.shape[0] == y_scored.shape[0],
    "train_rows_match_y_nonscored": train_features_clean.shape[0] == y_nonscored.shape[0],
    "train_rows_match_train_drug": train_features_clean.shape[0] == train_drug_clean.shape[0],
    "test_rows_match_submission": test_features_clean.shape[0] == sample_submission.shape[0]
}

id_alignment_audit

{'train_sig_id_unique': True,
 'test_sig_id_unique': True,
 'y_scored_sig_id_unique': True,
 'y_nonscored_sig_id_unique': True,
 'train_drug_sig_id_unique': True,
 'sample_submission_sig_id_unique': True,
 'train_vs_y_scored_order_match': True,
 'train_vs_y_nonscored_order_match': True,
 'train_vs_drug_order_match': True,
 'test_vs_submission_order_match': True,
 'train_rows_match_y_scored': True,
 'train_rows_match_y_nonscored': True,
 'train_rows_match_train_drug': True,
 'test_rows_match_submission': True}

In [16]:
train_feature_columns_without_id = [col for col in train_features_clean.columns if col != ID_COL]
test_feature_columns_without_id = [col for col in test_features_clean.columns if col != ID_COL]

feature_column_alignment_audit = {
    "train_feature_count_without_id": len(train_feature_columns_without_id),
    "test_feature_count_without_id": len(test_feature_columns_without_id),
    "train_test_feature_columns_match": train_feature_columns_without_id == test_feature_columns_without_id,
    "metadata_columns_present": all(col in train_features_clean.columns for col in metadata_features),
    "gene_columns_present": all(col in train_features_clean.columns for col in gene_features),
    "cell_columns_present": all(col in train_features_clean.columns for col in cell_features),
    "sample_submission_target_count": sample_submission.shape[1] - 1,
    "scored_target_count": len(scored_target_features),
    "submission_target_columns_match_scored_targets": sample_submission.columns[1:].tolist() == scored_target_features
}

feature_column_alignment_audit

{'train_feature_count_without_id': 875,
 'test_feature_count_without_id': 875,
 'train_test_feature_columns_match': True,
 'metadata_columns_present': True,
 'gene_columns_present': True,
 'cell_columns_present': True,
 'sample_submission_target_count': 206,
 'scored_target_count': 206,
 'submission_target_columns_match_scored_targets': True}

In [17]:
missing_value_audit = pd.DataFrame({
    "dataset": [
        "train_features_clean",
        "test_features_clean",
        "y_scored",
        "y_nonscored",
        "train_drug_clean",
        "sample_submission"
    ],
    "missing_values": [
        int(train_features_clean.isna().sum().sum()),
        int(test_features_clean.isna().sum().sum()),
        int(y_scored.isna().sum().sum()),
        int(y_nonscored.isna().sum().sum()),
        int(train_drug_clean.isna().sum().sum()),
        int(sample_submission.isna().sum().sum())
    ]
})

missing_value_audit

,dataset,missing_values
0,train_features_clean,0
1,test_features_clean,0
2,y_scored,0
3,y_nonscored,0
4,train_drug_clean,0
5,sample_submission,0


In [18]:
assert id_alignment_audit["train_sig_id_unique"], "train_features_clean sig_id is not unique"
assert id_alignment_audit["test_sig_id_unique"], "test_features_clean sig_id is not unique"
assert id_alignment_audit["train_vs_y_scored_order_match"], "train and y_scored sig_id order mismatch"
assert id_alignment_audit["train_vs_y_nonscored_order_match"], "train and y_nonscored sig_id order mismatch"
assert id_alignment_audit["train_vs_drug_order_match"], "train and train_drug sig_id order mismatch"
assert id_alignment_audit["test_vs_submission_order_match"], "test and sample_submission sig_id order mismatch"

assert feature_column_alignment_audit["train_test_feature_columns_match"], "train/test feature columns do not match"
assert feature_column_alignment_audit["metadata_columns_present"], "metadata columns missing"
assert feature_column_alignment_audit["gene_columns_present"], "gene columns missing"
assert feature_column_alignment_audit["cell_columns_present"], "cell columns missing"
assert feature_column_alignment_audit["submission_target_columns_match_scored_targets"], "submission target columns do not match scored targets"

assert missing_value_audit["missing_values"].sum() == 0, "Missing values found in loaded clean data"

print("Step 2 hard checks passed successfully.")

Step 2 hard checks passed successfully.


In [19]:
data_shape_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_2_data_shape_audit.csv",
    index=False
)

feature_group_count_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_2_feature_group_count_audit.csv",
    index=False
)

pd.DataFrame([id_alignment_audit]).to_csv(
    EXPERIMENT_LOG_DIR / "step_2_id_alignment_audit.csv",
    index=False
)

pd.DataFrame([feature_column_alignment_audit]).to_csv(
    EXPERIMENT_LOG_DIR / "step_2_feature_column_alignment_audit.csv",
    index=False
)

missing_value_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_2_missing_value_audit.csv",
    index=False
)

print("Step 2 data loading and audit files saved successfully.")

Step 2 data loading and audit files saved successfully.


In [20]:
step_2_status = {
    "step": "Step 2 — Load Clean Interim Data",
    "status": "completed",
    "clean_data_loaded": True,
    "feature_groups_loaded": True,
    "id_alignment_checked": True,
    "feature_column_alignment_checked": True,
    "missing_values_checked": True,
    "hard_checks_passed": True,
    "ready_for_step_3": True,
    "next_step": "Step 3 — Feature Group Verification"
}

step_2_status

{'step': 'Step 2 — Load Clean Interim Data',
 'status': 'completed',
 'clean_data_loaded': True,
 'feature_groups_loaded': True,
 'id_alignment_checked': True,
 'feature_column_alignment_checked': True,
 'missing_values_checked': True,
 'hard_checks_passed': True,
 'ready_for_step_3': True,
 'next_step': 'Step 3 — Feature Group Verification'}

## Step 3 — Feature Group Verification and Forbidden Column Pre-Audit

This step verifies that all feature groups are correctly separated before feature engineering.

The goal is to make sure that:

- metadata, gene, and cell features are present;
- scored and nonscored targets are separated from input features;
- `drug_id` is not used as a baseline input feature;
- no forbidden or target-derived feature can accidentally enter the model input matrix.

In [21]:
feature_group_verification = pd.DataFrame({
    "group_name": [
        "ID column",
        "Metadata features",
        "Gene features",
        "Cell features",
        "Scored target features",
        "Nonscored target features"
    ],
    "expected_pattern": [
        "sig_id",
        "cp_type, cp_time, cp_dose",
        "columns starting with g-",
        "columns starting with c-",
        "columns from y_scored excluding sig_id",
        "columns from y_nonscored excluding sig_id"
    ],
    "count": [
        1,
        len(metadata_features),
        len(gene_features),
        len(cell_features),
        len(scored_target_features),
        len(nonscored_target_features)
    ],
    "status": [
        ID_COL in train_features_clean.columns,
        all(col in train_features_clean.columns for col in metadata_features),
        all(col in train_features_clean.columns for col in gene_features),
        all(col in train_features_clean.columns for col in cell_features),
        all(col in y_scored.columns for col in scored_target_features),
        all(col in y_nonscored.columns for col in nonscored_target_features)
    ]
})

feature_group_verification

,group_name,expected_pattern,count,status
0,ID column,sig_id,1,True
1,Metadata features,"cp_type, cp_time, cp_dose",3,True
2,Gene features,columns starting with g-,772,True
3,Cell features,columns starting with c-,100,True
4,Scored target features,columns from y_scored excluding sig_id,206,True
5,Nonscored target features,columns from y_nonscored excluding sig_id,402,True


In [22]:
gene_cell_naming_audit = pd.DataFrame({
    "check": [
        "All gene features start with g-",
        "All cell features start with c-",
        "No gene feature appears in cell features",
        "No cell feature appears in gene features",
        "Gene features exist in train",
        "Gene features exist in test",
        "Cell features exist in train",
        "Cell features exist in test"
    ],
    "passed": [
        all(col.startswith("g-") for col in gene_features),
        all(col.startswith("c-") for col in cell_features),
        len(set(gene_features).intersection(set(cell_features))) == 0,
        len(set(cell_features).intersection(set(gene_features))) == 0,
        all(col in train_features_clean.columns for col in gene_features),
        all(col in test_features_clean.columns for col in gene_features),
        all(col in train_features_clean.columns for col in cell_features),
        all(col in test_features_clean.columns for col in cell_features)
    ]
})

gene_cell_naming_audit

,check,passed
0,All gene features start with g-,True
1,All cell features start with c-,True
2,No gene feature appears in cell features,True
3,No cell feature appears in gene features,True
4,Gene features exist in train,True
5,Gene features exist in test,True
6,Cell features exist in train,True
7,Cell features exist in test,True


In [23]:
duplicate_column_audit = pd.DataFrame({
    "dataset": [
        "train_features_clean",
        "test_features_clean",
        "y_scored",
        "y_nonscored",
        "train_drug_clean"
    ],
    "duplicate_column_count": [
        train_features_clean.columns.duplicated().sum(),
        test_features_clean.columns.duplicated().sum(),
        y_scored.columns.duplicated().sum(),
        y_nonscored.columns.duplicated().sum(),
        train_drug_clean.columns.duplicated().sum()
    ]
})

duplicate_column_audit

,dataset,duplicate_column_count
0,train_features_clean,0
1,test_features_clean,0
2,y_scored,0
3,y_nonscored,0
4,train_drug_clean,0


In [24]:
forbidden_input_columns = set(scored_target_features) | set(nonscored_target_features) | {
    "drug_id",
    "active_target_count",
    "zero_label_flag",
    "multi_label_count",
    "scored_positive_count",
    "nonscored_positive_count",
    "target_frequency",
    "fold_positive_count",
    "validation_positive_count",
    "target_cooccurrence_count"
}

forbidden_input_columns_preview = pd.DataFrame({
    "forbidden_column_or_pattern": sorted(list(forbidden_input_columns))[:30]
})

forbidden_input_columns_preview

,forbidden_column_or_pattern
0,11-beta-hsd1_inhibitor
1,5-alpha_reductase_inhibitor
2,abc_transporter_expression_enhancer
3,abl_inhibitor
4,acat_inhibitor
5,ace_inhibitor
6,acetylcholine_receptor_agonist
7,acetylcholine_receptor_antagonist
8,acetylcholine_release_enhancer
9,acetylcholinesterase_inhibitor


In [25]:
train_forbidden_columns_found = sorted(
    list(set(train_features_clean.columns).intersection(forbidden_input_columns))
)

test_forbidden_columns_found = sorted(
    list(set(test_features_clean.columns).intersection(forbidden_input_columns))
)

forbidden_column_pre_audit = pd.DataFrame({
    "dataset": [
        "train_features_clean",
        "test_features_clean"
    ],
    "forbidden_columns_found_count": [
        len(train_forbidden_columns_found),
        len(test_forbidden_columns_found)
    ],
    "forbidden_columns_found": [
        train_forbidden_columns_found,
        test_forbidden_columns_found
    ],
    "passed": [
        len(train_forbidden_columns_found) == 0,
        len(test_forbidden_columns_found) == 0
    ]
})

forbidden_column_pre_audit

,dataset,forbidden_columns_found_count,forbidden_columns_found,passed
0,train_features_clean,0,[],True
1,test_features_clean,0,[],True


In [26]:
target_separation_audit = pd.DataFrame({
    "check": [
        "No scored target appears in train_features_clean",
        "No scored target appears in test_features_clean",
        "No nonscored target appears in train_features_clean",
        "No nonscored target appears in test_features_clean",
        "Scored targets exist only in y_scored",
        "Nonscored targets exist only in y_nonscored"
    ],
    "passed": [
        len(set(scored_target_features).intersection(train_features_clean.columns)) == 0,
        len(set(scored_target_features).intersection(test_features_clean.columns)) == 0,
        len(set(nonscored_target_features).intersection(train_features_clean.columns)) == 0,
        len(set(nonscored_target_features).intersection(test_features_clean.columns)) == 0,
        all(col in y_scored.columns for col in scored_target_features),
        all(col in y_nonscored.columns for col in nonscored_target_features)
    ]
})

target_separation_audit

,check,passed
0,No scored target appears in train_features_clean,True
1,No scored target appears in test_features_clean,True
2,No nonscored target appears in train_features_...,True
3,No nonscored target appears in test_features_c...,True
4,Scored targets exist only in y_scored,True
5,Nonscored targets exist only in y_nonscored,True


In [27]:
drug_id_handling_audit = pd.DataFrame({
    "check": [
        "drug_id exists in train_drug_clean",
        "drug_id does not exist in train_features_clean",
        "drug_id does not exist in test_features_clean",
        "drug_id will not be used in baseline feature matrices",
        "drug_id is available only for validation diagnostics"
    ],
    "passed": [
        "drug_id" in train_drug_clean.columns,
        "drug_id" not in train_features_clean.columns,
        "drug_id" not in test_features_clean.columns,
        True,
        True
    ],
    "reason": [
        "Needed for drug-overlap and group-validation diagnostics",
        "Prevents accidental baseline input leakage",
        "Test-side standard feature matrix should not depend on drug_id",
        "Baseline X should use metadata + gene + cell + deterministic summaries only",
        "Drug-aware validation can be tested later as robustness analysis"
    ]
})

drug_id_handling_audit

,check,passed,reason
0,drug_id exists in train_drug_clean,True,Needed for drug-overlap and group-validation d...
1,drug_id does not exist in train_features_clean,True,Prevents accidental baseline input leakage
2,drug_id does not exist in test_features_clean,True,Test-side standard feature matrix should not d...
3,drug_id will not be used in baseline feature m...,True,Baseline X should use metadata + gene + cell +...
4,drug_id is available only for validation diagn...,True,Drug-aware validation can be tested later as r...


In [28]:
assert feature_group_verification["status"].all(), "Some feature groups are missing or invalid"
assert gene_cell_naming_audit["passed"].all(), "Gene/cell feature naming audit failed"
assert duplicate_column_audit["duplicate_column_count"].sum() == 0, "Duplicate columns found"
assert forbidden_column_pre_audit["passed"].all(), "Forbidden columns found in raw input features"
assert target_separation_audit["passed"].all(), "Target separation audit failed"
assert drug_id_handling_audit["passed"].all(), "Drug ID handling audit failed"

print("Step 3 feature group verification and forbidden-column pre-audit passed successfully.")

Step 3 feature group verification and forbidden-column pre-audit passed successfully.


In [29]:
feature_group_verification.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_feature_group_verification.csv",
    index=False
)

gene_cell_naming_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_gene_cell_naming_audit.csv",
    index=False
)

duplicate_column_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_duplicate_column_audit.csv",
    index=False
)

forbidden_column_pre_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_forbidden_column_pre_audit.csv",
    index=False
)

target_separation_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_target_separation_audit.csv",
    index=False
)

drug_id_handling_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_3_drug_id_handling_audit.csv",
    index=False
)

print("Step 3 audit files saved successfully.")

Step 3 audit files saved successfully.


In [30]:
step_3_status = {
    "step": "Step 3 — Feature Group Verification and Forbidden Column Pre-Audit",
    "status": "completed",
    "feature_groups_verified": True,
    "gene_cell_features_verified": True,
    "duplicate_columns_checked": True,
    "target_separation_checked": True,
    "forbidden_columns_checked": True,
    "drug_id_excluded_from_baseline": True,
    "ready_for_step_4": True,
    "next_step": "Step 4 — Build FE_V0 Raw Baseline Matrix"
}

step_3_status

{'step': 'Step 3 — Feature Group Verification and Forbidden Column Pre-Audit',
 'status': 'completed',
 'feature_groups_verified': True,
 'gene_cell_features_verified': True,
 'duplicate_columns_checked': True,
 'target_separation_checked': True,
 'forbidden_columns_checked': True,
 'drug_id_excluded_from_baseline': True,
 'ready_for_step_4': True,
 'next_step': 'Step 4 — Build FE_V0 Raw Baseline Matrix'}

## Step 4 — Build FE_V0 Raw Baseline Matrix

FE_V0 is the first leakage-free baseline feature set.

It uses only:

- metadata features: `cp_type`, `cp_time`, `cp_dose`
- raw gene features: `g-*`
- raw cell features: `c-*`

No target-derived features, nonscored targets, scored targets, drug ID, scaler, PCA, quantile transformation, or feature selection is used here.

Two versions are saved:

- `FE_V0_raw`: original metadata + gene + cell features
- `FE_V0`: model-ready matrix with deterministic one-hot metadata + gene + cell features

In [31]:
fe_v0_raw_input_columns = metadata_features + gene_features + cell_features
fe_v0_raw_columns = [ID_COL] + fe_v0_raw_input_columns

fe_v0_column_definition_audit = pd.DataFrame({
    "component": [
        "ID column",
        "metadata features",
        "gene features",
        "cell features",
        "total raw input features",
        "total raw columns including ID"
    ],
    "count": [
        1,
        len(metadata_features),
        len(gene_features),
        len(cell_features),
        len(fe_v0_raw_input_columns),
        len(fe_v0_raw_columns)
    ]
})

fe_v0_column_definition_audit

,component,count
0,ID column,1
1,metadata features,3
2,gene features,772
3,cell features,100
4,total raw input features,875
5,total raw columns including ID,876


In [32]:
missing_train_fe_v0_columns = sorted(
    list(set(fe_v0_raw_columns) - set(train_features_clean.columns))
)

missing_test_fe_v0_columns = sorted(
    list(set(fe_v0_raw_columns) - set(test_features_clean.columns))
)

fe_v0_missing_column_audit = pd.DataFrame({
    "dataset": [
        "train_features_clean",
        "test_features_clean"
    ],
    "missing_column_count": [
        len(missing_train_fe_v0_columns),
        len(missing_test_fe_v0_columns)
    ],
    "missing_columns": [
        missing_train_fe_v0_columns,
        missing_test_fe_v0_columns
    ],
    "passed": [
        len(missing_train_fe_v0_columns) == 0,
        len(missing_test_fe_v0_columns) == 0
    ]
})

fe_v0_missing_column_audit

,dataset,missing_column_count,missing_columns,passed
0,train_features_clean,0,[],True
1,test_features_clean,0,[],True


In [33]:
X_train_fe_v0_raw = train_features_clean[fe_v0_raw_columns].copy()
X_test_fe_v0_raw = test_features_clean[fe_v0_raw_columns].copy()

print("X_train_fe_v0_raw shape:", X_train_fe_v0_raw.shape)
print("X_test_fe_v0_raw shape :", X_test_fe_v0_raw.shape)

X_train_fe_v0_raw shape: (23814, 876)
X_test_fe_v0_raw shape : (3982, 876)


In [34]:
metadata_expected_values = {
    "cp_type": ["ctl_vehicle", "trt_cp"],
    "cp_time": ["24", "48", "72"],
    "cp_dose": ["D1", "D2"]
}

metadata_category_audit_rows = []

for col, expected_values in metadata_expected_values.items():
    train_values = sorted(X_train_fe_v0_raw[col].astype(str).unique().tolist())
    test_values = sorted(X_test_fe_v0_raw[col].astype(str).unique().tolist())
    
    metadata_category_audit_rows.append({
        "metadata_column": col,
        "expected_values": expected_values,
        "train_values": train_values,
        "test_values": test_values,
        "unexpected_train_values": sorted(list(set(train_values) - set(expected_values))),
        "unexpected_test_values": sorted(list(set(test_values) - set(expected_values))),
        "passed": (
            len(set(train_values) - set(expected_values)) == 0 and
            len(set(test_values) - set(expected_values)) == 0
        )
    })

metadata_category_audit = pd.DataFrame(metadata_category_audit_rows)

metadata_category_audit

,metadata_column,expected_values,train_values,test_values,unexpected_train_values,unexpected_test_values,passed
0,cp_type,"[ctl_vehicle, trt_cp]","[ctl_vehicle, trt_cp]","[ctl_vehicle, trt_cp]",[],[],True
1,cp_time,"[24, 48, 72]","[24, 48, 72]","[24, 48, 72]",[],[],True
2,cp_dose,"[D1, D2]","[D1, D2]","[D1, D2]",[],[],True


In [35]:
def build_metadata_ohe(df, id_col=ID_COL):
    """
    Deterministic one-hot metadata encoding for FE_V0.
    Uses fixed known MoA metadata categories.
    """
    meta = df[[id_col] + metadata_features].copy()
    
    meta["cp_type"] = meta["cp_type"].astype(str)
    meta["cp_time"] = meta["cp_time"].astype(int).astype(str)
    meta["cp_dose"] = meta["cp_dose"].astype(str)
    
    cp_type_ohe = pd.get_dummies(meta["cp_type"], prefix="cp_type", dtype=np.int8)
    cp_time_ohe = pd.get_dummies(meta["cp_time"], prefix="cp_time", dtype=np.int8)
    cp_dose_ohe = pd.get_dummies(meta["cp_dose"], prefix="cp_dose", dtype=np.int8)
    
    expected_ohe_columns = (
        [f"cp_type_{value}" for value in metadata_expected_values["cp_type"]] +
        [f"cp_time_{value}" for value in metadata_expected_values["cp_time"]] +
        [f"cp_dose_{value}" for value in metadata_expected_values["cp_dose"]]
    )
    
    meta_ohe = pd.concat(
        [meta[[id_col]], cp_type_ohe, cp_time_ohe, cp_dose_ohe],
        axis=1
    )
    
    meta_ohe = meta_ohe.reindex(
        columns=[id_col] + expected_ohe_columns,
        fill_value=0
    )
    
    return meta_ohe, expected_ohe_columns


train_metadata_ohe, metadata_ohe_columns = build_metadata_ohe(X_train_fe_v0_raw)
test_metadata_ohe, _ = build_metadata_ohe(X_test_fe_v0_raw)

print("train_metadata_ohe shape:", train_metadata_ohe.shape)
print("test_metadata_ohe shape :", test_metadata_ohe.shape)
print("metadata_ohe_columns:", metadata_ohe_columns)

train_metadata_ohe shape: (23814, 8)
test_metadata_ohe shape : (3982, 8)
metadata_ohe_columns: ['cp_type_ctl_vehicle', 'cp_type_trt_cp', 'cp_time_24', 'cp_time_48', 'cp_time_72', 'cp_dose_D1', 'cp_dose_D2']


In [36]:
train_gene_cell_raw = X_train_fe_v0_raw[[ID_COL] + gene_features + cell_features].copy()
test_gene_cell_raw = X_test_fe_v0_raw[[ID_COL] + gene_features + cell_features].copy()

X_train_fe_v0 = train_metadata_ohe.merge(
    train_gene_cell_raw,
    on=ID_COL,
    how="left",
    validate="one_to_one"
)

X_test_fe_v0 = test_metadata_ohe.merge(
    test_gene_cell_raw,
    on=ID_COL,
    how="left",
    validate="one_to_one"
)

print("X_train_fe_v0 shape:", X_train_fe_v0.shape)
print("X_test_fe_v0 shape :", X_test_fe_v0.shape)

X_train_fe_v0 shape: (23814, 880)
X_test_fe_v0 shape : (3982, 880)


In [37]:
fe_v0_train_columns_without_id = [col for col in X_train_fe_v0.columns if col != ID_COL]
fe_v0_test_columns_without_id = [col for col in X_test_fe_v0.columns if col != ID_COL]

fe_v0_alignment_audit = {
    "train_rows": X_train_fe_v0.shape[0],
    "test_rows": X_test_fe_v0.shape[0],
    "train_columns": X_train_fe_v0.shape[1],
    "test_columns": X_test_fe_v0.shape[1],
    "train_test_columns_match": fe_v0_train_columns_without_id == fe_v0_test_columns_without_id,
    "train_sig_id_order_preserved": X_train_fe_v0[ID_COL].tolist() == train_features_clean[ID_COL].tolist(),
    "test_sig_id_order_preserved": X_test_fe_v0[ID_COL].tolist() == test_features_clean[ID_COL].tolist(),
    "metadata_ohe_feature_count": len(metadata_ohe_columns),
    "gene_feature_count": len(gene_features),
    "cell_feature_count": len(cell_features),
    "total_model_ready_features_without_id": len(fe_v0_train_columns_without_id)
}

fe_v0_alignment_audit

{'train_rows': 23814,
 'test_rows': 3982,
 'train_columns': 880,
 'test_columns': 880,
 'train_test_columns_match': True,
 'train_sig_id_order_preserved': True,
 'test_sig_id_order_preserved': True,
 'metadata_ohe_feature_count': 7,
 'gene_feature_count': 772,
 'cell_feature_count': 100,
 'total_model_ready_features_without_id': 879}

In [38]:
non_numeric_train_fe_v0 = [
    col for col in X_train_fe_v0.columns
    if col != ID_COL and not pd.api.types.is_numeric_dtype(X_train_fe_v0[col])
]

non_numeric_test_fe_v0 = [
    col for col in X_test_fe_v0.columns
    if col != ID_COL and not pd.api.types.is_numeric_dtype(X_test_fe_v0[col])
]

fe_v0_dtype_audit = pd.DataFrame({
    "dataset": [
        "X_train_fe_v0",
        "X_test_fe_v0"
    ],
    "non_numeric_feature_count_excluding_id": [
        len(non_numeric_train_fe_v0),
        len(non_numeric_test_fe_v0)
    ],
    "non_numeric_features": [
        non_numeric_train_fe_v0,
        non_numeric_test_fe_v0
    ],
    "passed": [
        len(non_numeric_train_fe_v0) == 0,
        len(non_numeric_test_fe_v0) == 0
    ]
})

fe_v0_dtype_audit

,dataset,non_numeric_feature_count_excluding_id,non_numeric_features,passed
0,X_train_fe_v0,0,[],True
1,X_test_fe_v0,0,[],True


In [39]:
fe_v0_forbidden_train_found = sorted(
    list(set(X_train_fe_v0.columns).intersection(forbidden_input_columns))
)

fe_v0_forbidden_test_found = sorted(
    list(set(X_test_fe_v0.columns).intersection(forbidden_input_columns))
)

fe_v0_forbidden_column_audit = pd.DataFrame({
    "dataset": [
        "X_train_fe_v0",
        "X_test_fe_v0"
    ],
    "forbidden_columns_found_count": [
        len(fe_v0_forbidden_train_found),
        len(fe_v0_forbidden_test_found)
    ],
    "forbidden_columns_found": [
        fe_v0_forbidden_train_found,
        fe_v0_forbidden_test_found
    ],
    "passed": [
        len(fe_v0_forbidden_train_found) == 0,
        len(fe_v0_forbidden_test_found) == 0
    ]
})

fe_v0_forbidden_column_audit

,dataset,forbidden_columns_found_count,forbidden_columns_found,passed
0,X_train_fe_v0,0,[],True
1,X_test_fe_v0,0,[],True


In [40]:
assert fe_v0_missing_column_audit["passed"].all(), "Some FE_V0 raw columns are missing"
assert metadata_category_audit["passed"].all(), "Unexpected metadata category found"
assert fe_v0_alignment_audit["train_test_columns_match"], "FE_V0 train/test columns do not match"
assert fe_v0_alignment_audit["train_sig_id_order_preserved"], "FE_V0 train sig_id order changed"
assert fe_v0_alignment_audit["test_sig_id_order_preserved"], "FE_V0 test sig_id order changed"
assert fe_v0_dtype_audit["passed"].all(), "FE_V0 contains non-numeric model-ready features"
assert fe_v0_forbidden_column_audit["passed"].all(), "Forbidden columns found in FE_V0"

print("Step 4 FE_V0 hard checks passed successfully.")

Step 4 FE_V0 hard checks passed successfully.


In [41]:
# Save raw FE_V0
X_train_fe_v0_raw.to_parquet(
    DATA_PROCESSED_DIR / "X_train_FE_V0_raw.parquet",
    index=False
)

X_test_fe_v0_raw.to_parquet(
    DATA_PROCESSED_DIR / "X_test_FE_V0_raw.parquet",
    index=False
)

# Save model-ready FE_V0
X_train_fe_v0.to_parquet(
    DATA_PROCESSED_DIR / "X_train_FE_V0.parquet",
    index=False
)

X_test_fe_v0.to_parquet(
    DATA_PROCESSED_DIR / "X_test_FE_V0.parquet",
    index=False
)

# Save y copy for easy model-training handoff
y_scored.to_parquet(
    DATA_PROCESSED_DIR / "y_scored.parquet",
    index=False
)

# Save audits
fe_v0_column_definition_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_4_fe_v0_column_definition_audit.csv",
    index=False
)

fe_v0_missing_column_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_4_fe_v0_missing_column_audit.csv",
    index=False
)

metadata_category_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_4_metadata_category_audit.csv",
    index=False
)

pd.DataFrame([fe_v0_alignment_audit]).to_csv(
    EXPERIMENT_LOG_DIR / "step_4_fe_v0_alignment_audit.csv",
    index=False
)

fe_v0_dtype_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_4_fe_v0_dtype_audit.csv",
    index=False
)

fe_v0_forbidden_column_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_4_fe_v0_forbidden_column_audit.csv",
    index=False
)

print("Step 4 FE_V0 matrices and audit files saved successfully.")

Step 4 FE_V0 matrices and audit files saved successfully.


In [42]:
fe_v0_manifest = {
    "feature_set": "FE_V0",
    "description": "Raw baseline feature matrix with deterministic one-hot metadata plus raw gene and cell features.",
    "id_column": ID_COL,
    "raw_metadata_features": metadata_features,
    "metadata_ohe_features": metadata_ohe_columns,
    "gene_feature_count": len(gene_features),
    "cell_feature_count": len(cell_features),
    "model_ready_feature_count_excluding_id": len(fe_v0_train_columns_without_id),
    "train_shape": list(X_train_fe_v0.shape),
    "test_shape": list(X_test_fe_v0.shape),
    "forbidden_inputs_excluded": True,
    "scored_targets_excluded": True,
    "nonscored_targets_excluded": True,
    "drug_id_excluded": True,
    "learned_transformations_used": False,
    "saved_files": {
        "X_train_FE_V0_raw": str(DATA_PROCESSED_DIR / "X_train_FE_V0_raw.parquet"),
        "X_test_FE_V0_raw": str(DATA_PROCESSED_DIR / "X_test_FE_V0_raw.parquet"),
        "X_train_FE_V0": str(DATA_PROCESSED_DIR / "X_train_FE_V0.parquet"),
        "X_test_FE_V0": str(DATA_PROCESSED_DIR / "X_test_FE_V0.parquet"),
        "y_scored": str(DATA_PROCESSED_DIR / "y_scored.parquet")
    }
}

with open(DATA_PROCESSED_DIR / "FE_V0_manifest.json", "w", encoding="utf-8") as f:
    json.dump(fe_v0_manifest, f, indent=2)

fe_v0_manifest

{'feature_set': 'FE_V0',
 'description': 'Raw baseline feature matrix with deterministic one-hot metadata plus raw gene and cell features.',
 'id_column': 'sig_id',
 'raw_metadata_features': ['cp_type', 'cp_time', 'cp_dose'],
 'metadata_ohe_features': ['cp_type_ctl_vehicle',
  'cp_type_trt_cp',
  'cp_time_24',
  'cp_time_48',
  'cp_time_72',
  'cp_dose_D1',
  'cp_dose_D2'],
 'gene_feature_count': 772,
 'cell_feature_count': 100,
 'model_ready_feature_count_excluding_id': 879,
 'train_shape': [23814, 880],
 'test_shape': [3982, 880],
 'forbidden_inputs_excluded': True,
 'scored_targets_excluded': True,
 'nonscored_targets_excluded': True,
 'drug_id_excluded': True,
 'learned_transformations_used': False,
 'saved_files': {'X_train_FE_V0_raw': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\moa-prediction-drug-response\\data\\processed\\X_train_FE_V0_raw.parquet',
  'X_test_FE_V0_raw': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\moa-prediction

In [43]:
step_4_status = {
    "step": "Step 4 — Build FE_V0 Raw Baseline Matrix",
    "status": "completed",
    "fe_v0_raw_saved": True,
    "fe_v0_model_ready_saved": True,
    "metadata_encoded_deterministically": True,
    "train_test_columns_match": fe_v0_alignment_audit["train_test_columns_match"],
    "no_forbidden_columns": fe_v0_forbidden_column_audit["passed"].all(),
    "no_learned_transformations_used": True,
    "ready_for_step_5": True,
    "next_step": "Step 5 — FE_V0 Audit and Baseline Handoff"
}

step_4_status

{'step': 'Step 4 — Build FE_V0 Raw Baseline Matrix',
 'status': 'completed',
 'fe_v0_raw_saved': True,
 'fe_v0_model_ready_saved': True,
 'metadata_encoded_deterministically': True,
 'train_test_columns_match': True,
 'no_forbidden_columns': np.True_,
 'no_learned_transformations_used': True,
 'ready_for_step_5': True,
 'next_step': 'Step 5 — FE_V0 Audit and Baseline Handoff'}

## Step 5 — FE_V0 Audit and Baseline Handoff

This step audits the saved FE_V0 feature matrices and prepares the baseline handoff for model training.

No new feature engineering is performed here.

The goals are:

- verify FE_V0 saved artifacts exist;
- verify train/test schema alignment;
- verify ID order and target alignment;
- check missing, infinite, non-numeric, and constant-feature issues;
- confirm metadata one-hot encoding validity;
- save a compact handoff registry for model training.

In [45]:
fe_v0_artifact_paths = {
    "X_train_FE_V0_raw": DATA_PROCESSED_DIR / "X_train_FE_V0_raw.parquet",
    "X_test_FE_V0_raw": DATA_PROCESSED_DIR / "X_test_FE_V0_raw.parquet",
    "X_train_FE_V0": DATA_PROCESSED_DIR / "X_train_FE_V0.parquet",
    "X_test_FE_V0": DATA_PROCESSED_DIR / "X_test_FE_V0.parquet",
    "y_scored": DATA_PROCESSED_DIR / "y_scored.parquet",
    "FE_V0_manifest": DATA_PROCESSED_DIR / "FE_V0_manifest.json"
}

fe_v0_artifact_existence_audit = pd.DataFrame({
    "artifact": list(fe_v0_artifact_paths.keys()),
    "path": [str(path) for path in fe_v0_artifact_paths.values()],
    "exists": [path.exists() for path in fe_v0_artifact_paths.values()]
})

fe_v0_artifact_existence_audit

,artifact,path,exists
0,X_train_FE_V0_raw,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
1,X_test_FE_V0_raw,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
2,X_train_FE_V0,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
3,X_test_FE_V0,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
4,y_scored,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True
5,FE_V0_manifest,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...,True


In [46]:
missing_fe_v0_artifacts = fe_v0_artifact_existence_audit.loc[
    fe_v0_artifact_existence_audit["exists"] == False,
    ["artifact", "path"]
]

if len(missing_fe_v0_artifacts) > 0:
    display(missing_fe_v0_artifacts)
    raise FileNotFoundError("Some FE_V0 artifacts are missing. Re-run Step 4 before continuing.")

print("All FE_V0 artifacts exist.")

All FE_V0 artifacts exist.


In [47]:
X_train_FE_V0_saved = pd.read_parquet(fe_v0_artifact_paths["X_train_FE_V0"])
X_test_FE_V0_saved = pd.read_parquet(fe_v0_artifact_paths["X_test_FE_V0"])
y_scored_saved = pd.read_parquet(fe_v0_artifact_paths["y_scored"])

with open(fe_v0_artifact_paths["FE_V0_manifest"], "r", encoding="utf-8") as f:
    FE_V0_manifest_saved = json.load(f)

print("Saved FE_V0 artifacts reloaded successfully.")
print("X_train_FE_V0_saved:", X_train_FE_V0_saved.shape)
print("X_test_FE_V0_saved :", X_test_FE_V0_saved.shape)
print("y_scored_saved     :", y_scored_saved.shape)

Saved FE_V0 artifacts reloaded successfully.
X_train_FE_V0_saved: (23814, 880)
X_test_FE_V0_saved : (3982, 880)
y_scored_saved     : (23814, 207)


In [48]:
fe_v0_saved_memory_consistency_audit = pd.DataFrame({
    "check": [
        "Saved train shape matches in-memory train shape",
        "Saved test shape matches in-memory test shape",
        "Saved train columns match in-memory train columns",
        "Saved test columns match in-memory test columns",
        "Saved train sig_id order matches in-memory train sig_id order",
        "Saved test sig_id order matches in-memory test sig_id order",
        "Saved y_scored shape matches in-memory y_scored shape",
        "Saved y_scored columns match in-memory y_scored columns"
    ],
    "passed": [
        X_train_FE_V0_saved.shape == X_train_fe_v0.shape,
        X_test_FE_V0_saved.shape == X_test_fe_v0.shape,
        X_train_FE_V0_saved.columns.tolist() == X_train_fe_v0.columns.tolist(),
        X_test_FE_V0_saved.columns.tolist() == X_test_fe_v0.columns.tolist(),
        X_train_FE_V0_saved[ID_COL].tolist() == X_train_fe_v0[ID_COL].tolist(),
        X_test_FE_V0_saved[ID_COL].tolist() == X_test_fe_v0[ID_COL].tolist(),
        y_scored_saved.shape == y_scored.shape,
        y_scored_saved.columns.tolist() == y_scored.columns.tolist()
    ]
})

fe_v0_saved_memory_consistency_audit

,check,passed
0,Saved train shape matches in-memory train shape,True
1,Saved test shape matches in-memory test shape,True
2,Saved train columns match in-memory train columns,True
3,Saved test columns match in-memory test columns,True
4,Saved train sig_id order matches in-memory tra...,True
5,Saved test sig_id order matches in-memory test...,True
6,Saved y_scored shape matches in-memory y_score...,True
7,Saved y_scored columns match in-memory y_score...,True


In [49]:
fe_v0_model_features = [col for col in X_train_FE_V0_saved.columns if col != ID_COL]

fe_v0_feature_composition_audit = pd.DataFrame({
    "component": [
        "ID column",
        "metadata one-hot features",
        "raw gene features",
        "raw cell features",
        "total model features excluding ID",
        "total columns including ID"
    ],
    "count": [
        1,
        len(metadata_ohe_columns),
        len(gene_features),
        len(cell_features),
        len(fe_v0_model_features),
        X_train_FE_V0_saved.shape[1]
    ]
})

fe_v0_feature_composition_audit

,component,count
0,ID column,1
1,metadata one-hot features,7
2,raw gene features,772
3,raw cell features,100
4,total model features excluding ID,879
5,total columns including ID,880


In [50]:
def finite_value_audit(df, dataset_name, id_col=ID_COL):
    feature_cols = [col for col in df.columns if col != id_col]
    numeric_values = df[feature_cols].to_numpy(dtype=np.float64)
    
    return {
        "dataset": dataset_name,
        "rows": df.shape[0],
        "features_excluding_id": len(feature_cols),
        "missing_values": int(df[feature_cols].isna().sum().sum()),
        "positive_infinite_values": int(np.isposinf(numeric_values).sum()),
        "negative_infinite_values": int(np.isneginf(numeric_values).sum()),
        "all_values_finite": bool(np.isfinite(numeric_values).all())
    }


fe_v0_missing_infinite_audit = pd.DataFrame([
    finite_value_audit(X_train_FE_V0_saved, "X_train_FE_V0"),
    finite_value_audit(X_test_FE_V0_saved, "X_test_FE_V0")
])

fe_v0_missing_infinite_audit

,dataset,rows,features_excluding_id,missing_values,positive_infinite_values,negative_infinite_values,all_values_finite
0,X_train_FE_V0,23814,879,0,0,0,True
1,X_test_FE_V0,3982,879,0,0,0,True


In [51]:
train_feature_nunique = X_train_FE_V0_saved[fe_v0_model_features].nunique(dropna=False)
test_feature_nunique = X_test_FE_V0_saved[fe_v0_model_features].nunique(dropna=False)

constant_train_features = train_feature_nunique[train_feature_nunique <= 1].index.tolist()
constant_test_features = test_feature_nunique[test_feature_nunique <= 1].index.tolist()

fe_v0_constant_feature_audit = pd.DataFrame({
    "dataset": [
        "X_train_FE_V0",
        "X_test_FE_V0"
    ],
    "constant_feature_count": [
        len(constant_train_features),
        len(constant_test_features)
    ],
    "constant_features_preview": [
        constant_train_features[:20],
        constant_test_features[:20]
    ],
    "decision": [
        "Do not drop here; decide inside modeling pipeline if needed",
        "Do not drop here; train-side pipeline must control feature selection"
    ]
})

fe_v0_constant_feature_audit

,dataset,constant_feature_count,constant_features_preview,decision
0,X_train_FE_V0,0,[],Do not drop here; decide inside modeling pipel...
1,X_test_FE_V0,0,[],Do not drop here; train-side pipeline must con...


In [52]:
metadata_ohe_groups = {
    "cp_type": [col for col in metadata_ohe_columns if col.startswith("cp_type_")],
    "cp_time": [col for col in metadata_ohe_columns if col.startswith("cp_time_")],
    "cp_dose": [col for col in metadata_ohe_columns if col.startswith("cp_dose_")]
}

metadata_ohe_integrity_rows = []

for dataset_name, df in [
    ("X_train_FE_V0", X_train_FE_V0_saved),
    ("X_test_FE_V0", X_test_FE_V0_saved)
]:
    for group_name, group_cols in metadata_ohe_groups.items():
        row_sums = df[group_cols].sum(axis=1)
        
        metadata_ohe_integrity_rows.append({
            "dataset": dataset_name,
            "metadata_group": group_name,
            "ohe_columns": group_cols,
            "row_sum_min": int(row_sums.min()),
            "row_sum_max": int(row_sums.max()),
            "invalid_row_count": int((row_sums != 1).sum()),
            "passed": bool((row_sums == 1).all())
        })

fe_v0_metadata_ohe_integrity_audit = pd.DataFrame(metadata_ohe_integrity_rows)

fe_v0_metadata_ohe_integrity_audit

,dataset,metadata_group,ohe_columns,row_sum_min,row_sum_max,invalid_row_count,passed
0,X_train_FE_V0,cp_type,"[cp_type_ctl_vehicle, cp_type_trt_cp]",1,1,0,True
1,X_train_FE_V0,cp_time,"[cp_time_24, cp_time_48, cp_time_72]",1,1,0,True
2,X_train_FE_V0,cp_dose,"[cp_dose_D1, cp_dose_D2]",1,1,0,True
3,X_test_FE_V0,cp_type,"[cp_type_ctl_vehicle, cp_type_trt_cp]",1,1,0,True
4,X_test_FE_V0,cp_time,"[cp_time_24, cp_time_48, cp_time_72]",1,1,0,True
5,X_test_FE_V0,cp_dose,"[cp_dose_D1, cp_dose_D2]",1,1,0,True


In [53]:
fe_v0_target_handoff_audit = pd.DataFrame({
    "check": [
        "X_train_FE_V0 rows match y_scored rows",
        "X_train_FE_V0 sig_id order matches y_scored sig_id order",
        "y_scored has expected target count",
        "scored target columns match stored scored_target_features",
        "X_test_FE_V0 sig_id order matches sample_submission sig_id order",
        "sample_submission target columns match scored targets"
    ],
    "passed": [
        X_train_FE_V0_saved.shape[0] == y_scored_saved.shape[0],
        X_train_FE_V0_saved[ID_COL].tolist() == y_scored_saved[ID_COL].tolist(),
        len([col for col in y_scored_saved.columns if col != ID_COL]) == len(scored_target_features),
        y_scored_saved.columns[1:].tolist() == scored_target_features,
        X_test_FE_V0_saved[ID_COL].tolist() == sample_submission[ID_COL].tolist(),
        sample_submission.columns[1:].tolist() == scored_target_features
    ]
})

fe_v0_target_handoff_audit

,check,passed
0,X_train_FE_V0 rows match y_scored rows,True
1,X_train_FE_V0 sig_id order matches y_scored si...,True
2,y_scored has expected target count,True
3,scored target columns match stored scored_targ...,True
4,X_test_FE_V0 sig_id order matches sample_submi...,True
5,sample_submission target columns match scored ...,True


In [54]:
fe_v0_baseline_handoff_summary = pd.DataFrame({
    "item": [
        "feature_set_name",
        "train_feature_file",
        "test_feature_file",
        "target_file",
        "train_shape",
        "test_shape",
        "target_shape",
        "model_features_excluding_id",
        "metadata_ohe_features",
        "gene_features",
        "cell_features",
        "recommended_first_model",
        "recommended_validation",
        "control_row_rule"
    ],
    "value": [
        "FE_V0",
        str(fe_v0_artifact_paths["X_train_FE_V0"]),
        str(fe_v0_artifact_paths["X_test_FE_V0"]),
        str(fe_v0_artifact_paths["y_scored"]),
        str(tuple(X_train_FE_V0_saved.shape)),
        str(tuple(X_test_FE_V0_saved.shape)),
        str(tuple(y_scored_saved.shape)),
        len(fe_v0_model_features),
        len(metadata_ohe_columns),
        len(gene_features),
        len(cell_features),
        "multi-label logistic baseline or tree boosting baseline",
        "random/multilabel-aware CV first; drug-aware GroupKFold later as robustness",
        "set ctl_vehicle predictions to zero only after model prediction"
    ]
})

fe_v0_baseline_handoff_summary

,item,value
0,feature_set_name,FE_V0
1,train_feature_file,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
2,test_feature_file,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
3,target_file,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\M...
4,train_shape,"(23814, 880)"
5,test_shape,"(3982, 880)"
6,target_shape,"(23814, 207)"
7,model_features_excluding_id,879
8,metadata_ohe_features,7
9,gene_features,772


In [55]:
assert fe_v0_saved_memory_consistency_audit["passed"].all(), "Saved FE_V0 artifacts do not match in-memory FE_V0"
assert fe_v0_missing_infinite_audit["all_values_finite"].all(), "FE_V0 contains missing or infinite values"
assert fe_v0_metadata_ohe_integrity_audit["passed"].all(), "Metadata one-hot integrity check failed"
assert fe_v0_target_handoff_audit["passed"].all(), "FE_V0 target handoff audit failed"

print("Step 5 FE_V0 audit and baseline handoff checks passed successfully.")

Step 5 FE_V0 audit and baseline handoff checks passed successfully.


In [56]:
fe_v0_artifact_existence_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_5_fe_v0_artifact_existence_audit.csv",
    index=False
)

fe_v0_saved_memory_consistency_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_5_fe_v0_saved_memory_consistency_audit.csv",
    index=False
)

fe_v0_feature_composition_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_5_fe_v0_feature_composition_audit.csv",
    index=False
)

fe_v0_missing_infinite_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_5_fe_v0_missing_infinite_audit.csv",
    index=False
)

fe_v0_constant_feature_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_5_fe_v0_constant_feature_audit.csv",
    index=False
)

fe_v0_metadata_ohe_integrity_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_5_fe_v0_metadata_ohe_integrity_audit.csv",
    index=False
)

fe_v0_target_handoff_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_5_fe_v0_target_handoff_audit.csv",
    index=False
)

fe_v0_baseline_handoff_summary.to_csv(
    EXPERIMENT_LOG_DIR / "step_5_fe_v0_baseline_handoff_summary.csv",
    index=False
)

print("Step 5 FE_V0 audit and handoff files saved successfully.")

Step 5 FE_V0 audit and handoff files saved successfully.


In [57]:
feature_set_registry = {
    "available_feature_sets": {
        "FE_V0": {
            "description": "Deterministic one-hot metadata + raw gene + raw cell features.",
            "train_path": str(fe_v0_artifact_paths["X_train_FE_V0"]),
            "test_path": str(fe_v0_artifact_paths["X_test_FE_V0"]),
            "target_path": str(fe_v0_artifact_paths["y_scored"]),
            "manifest_path": str(fe_v0_artifact_paths["FE_V0_manifest"]),
            "train_shape": list(X_train_FE_V0_saved.shape),
            "test_shape": list(X_test_FE_V0_saved.shape),
            "target_shape": list(y_scored_saved.shape),
            "feature_count_excluding_id": len(fe_v0_model_features),
            "leakage_safe": True,
            "learned_transformations_used": False,
            "ready_for_model_training": True
        }
    },
    "next_planned_feature_sets": {
        "FE_V1": "FE_V0 + row-wise gene/cell summary features",
        "FE_V2": "FE_V1 + small audited gene-cell interaction features",
        "FE_V3": "Optional metadata interaction features"
    }
}

with open(DATA_PROCESSED_DIR / "feature_set_registry.json", "w", encoding="utf-8") as f:
    json.dump(feature_set_registry, f, indent=2)

feature_set_registry

{'available_feature_sets': {'FE_V0': {'description': 'Deterministic one-hot metadata + raw gene + raw cell features.',
   'train_path': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\moa-prediction-drug-response\\data\\processed\\X_train_FE_V0.parquet',
   'test_path': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\moa-prediction-drug-response\\data\\processed\\X_test_FE_V0.parquet',
   'target_path': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\moa-prediction-drug-response\\data\\processed\\y_scored.parquet',
   'manifest_path': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\moa-prediction-drug-response\\data\\processed\\FE_V0_manifest.json',
   'train_shape': [23814, 880],
   'test_shape': [3982, 880],
   'target_shape': [23814, 207],
   'feature_count_excluding_id': 879,
   'leakage_safe': True,
   'learned_transformations_used': False,
   'ready_for_model_training': True}},
 'next_planned_feat

In [58]:
step_5_status = {
    "step": "Step 5 — FE_V0 Audit and Baseline Handoff",
    "status": "completed",
    "fe_v0_artifacts_exist": bool(fe_v0_artifact_existence_audit["exists"].all()),
    "saved_memory_consistency_passed": bool(fe_v0_saved_memory_consistency_audit["passed"].all()),
    "missing_infinite_check_passed": bool(fe_v0_missing_infinite_audit["all_values_finite"].all()),
    "metadata_ohe_integrity_passed": bool(fe_v0_metadata_ohe_integrity_audit["passed"].all()),
    "target_handoff_passed": bool(fe_v0_target_handoff_audit["passed"].all()),
    "feature_set_registry_saved": True,
    "ready_for_step_6": True,
    "next_step": "Step 6 — Build FE_V1 Row-wise Gene/Cell Summary Features"
}

step_5_status

{'step': 'Step 5 — FE_V0 Audit and Baseline Handoff',
 'status': 'completed',
 'fe_v0_artifacts_exist': True,
 'saved_memory_consistency_passed': True,
 'missing_infinite_check_passed': True,
 'metadata_ohe_integrity_passed': True,
 'target_handoff_passed': True,
 'feature_set_registry_saved': True,
 'ready_for_step_6': True,
 'next_step': 'Step 6 — Build FE_V1 Row-wise Gene/Cell Summary Features'}

## Step 6 — Build FE_V1 Row-wise Gene/Cell Summary Features

FE_V1 extends FE_V0 by adding deterministic row-wise summary features from raw gene and cell blocks.

Added features:

Gene summary features:

- gene_mean
- gene_std
- gene_abs_mean
- gene_min
- gene_max
- gene_range

Cell summary features:

- cell_mean
- cell_std
- cell_abs_mean
- cell_min
- cell_max
- cell_range

These features are leakage-safe because each value is computed only from the same sample's input features.

No target-derived feature, nonscored target, scored target, drug ID, scaler, PCA, quantile transformation, or feature selection is used in this step.

In [59]:
def build_rowwise_block_summary(df, block_features, block_name, id_col=ID_COL):
    """
    Build deterministic row-wise summary features for a feature block.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing ID column and block features.
    block_features : list
        Feature columns belonging to one block, e.g., gene_features or cell_features.
    block_name : str
        Prefix name, e.g., "gene" or "cell".
    id_col : str
        ID column name.
        
    Returns
    -------
    pd.DataFrame
        Dataframe with ID column + row-wise summary features.
    """
    block_values = df[block_features].astype(np.float32)
    
    summary_df = pd.DataFrame({
        id_col: df[id_col].values,
        f"{block_name}_mean": block_values.mean(axis=1).astype(np.float32),
        f"{block_name}_std": block_values.std(axis=1).astype(np.float32),
        f"{block_name}_abs_mean": block_values.abs().mean(axis=1).astype(np.float32),
        f"{block_name}_min": block_values.min(axis=1).astype(np.float32),
        f"{block_name}_max": block_values.max(axis=1).astype(np.float32),
    })
    
    summary_df[f"{block_name}_range"] = (
        summary_df[f"{block_name}_max"] - summary_df[f"{block_name}_min"]
    ).astype(np.float32)
    
    return summary_df

In [60]:
train_gene_summary = build_rowwise_block_summary(
    train_features_clean,
    gene_features,
    block_name="gene"
)

test_gene_summary = build_rowwise_block_summary(
    test_features_clean,
    gene_features,
    block_name="gene"
)

gene_summary_features = [col for col in train_gene_summary.columns if col != ID_COL]

print("train_gene_summary shape:", train_gene_summary.shape)
print("test_gene_summary shape :", test_gene_summary.shape)
print("gene_summary_features:", gene_summary_features)

train_gene_summary shape: (23814, 7)
test_gene_summary shape : (3982, 7)
gene_summary_features: ['gene_mean', 'gene_std', 'gene_abs_mean', 'gene_min', 'gene_max', 'gene_range']


In [61]:
train_cell_summary = build_rowwise_block_summary(
    train_features_clean,
    cell_features,
    block_name="cell"
)

test_cell_summary = build_rowwise_block_summary(
    test_features_clean,
    cell_features,
    block_name="cell"
)

cell_summary_features = [col for col in train_cell_summary.columns if col != ID_COL]

print("train_cell_summary shape:", train_cell_summary.shape)
print("test_cell_summary shape :", test_cell_summary.shape)
print("cell_summary_features:", cell_summary_features)

train_cell_summary shape: (23814, 7)
test_cell_summary shape : (3982, 7)
cell_summary_features: ['cell_mean', 'cell_std', 'cell_abs_mean', 'cell_min', 'cell_max', 'cell_range']


In [62]:
fe_v1_summary_feature_definition_audit = pd.DataFrame({
    "summary_feature": gene_summary_features + cell_summary_features,
    "source_block": ["gene"] * len(gene_summary_features) + ["cell"] * len(cell_summary_features),
    "calculation": [
        "row-wise mean across all gene features",
        "row-wise standard deviation across all gene features",
        "row-wise mean absolute value across all gene features",
        "row-wise minimum across all gene features",
        "row-wise maximum across all gene features",
        "row-wise max minus min across all gene features",
        "row-wise mean across all cell features",
        "row-wise standard deviation across all cell features",
        "row-wise mean absolute value across all cell features",
        "row-wise minimum across all cell features",
        "row-wise maximum across all cell features",
        "row-wise max minus min across all cell features"
    ],
    "leakage_status": ["safe_rowwise_input_only"] * (len(gene_summary_features) + len(cell_summary_features))
})

fe_v1_summary_feature_definition_audit

,summary_feature,source_block,calculation,leakage_status
0,gene_mean,gene,row-wise mean across all gene features,safe_rowwise_input_only
1,gene_std,gene,row-wise standard deviation across all gene fe...,safe_rowwise_input_only
2,gene_abs_mean,gene,row-wise mean absolute value across all gene f...,safe_rowwise_input_only
3,gene_min,gene,row-wise minimum across all gene features,safe_rowwise_input_only
4,gene_max,gene,row-wise maximum across all gene features,safe_rowwise_input_only
5,gene_range,gene,row-wise max minus min across all gene features,safe_rowwise_input_only
6,cell_mean,cell,row-wise mean across all cell features,safe_rowwise_input_only
7,cell_std,cell,row-wise standard deviation across all cell fe...,safe_rowwise_input_only
8,cell_abs_mean,cell,row-wise mean absolute value across all cell f...,safe_rowwise_input_only
9,cell_min,cell,row-wise minimum across all cell features,safe_rowwise_input_only


In [63]:
X_train_fe_v1 = (
    X_train_fe_v0
    .merge(train_gene_summary, on=ID_COL, how="left", validate="one_to_one")
    .merge(train_cell_summary, on=ID_COL, how="left", validate="one_to_one")
)

X_test_fe_v1 = (
    X_test_fe_v0
    .merge(test_gene_summary, on=ID_COL, how="left", validate="one_to_one")
    .merge(test_cell_summary, on=ID_COL, how="left", validate="one_to_one")
)

print("X_train_fe_v0 shape:", X_train_fe_v0.shape)
print("X_train_fe_v1 shape:", X_train_fe_v1.shape)
print("X_test_fe_v0 shape :", X_test_fe_v0.shape)
print("X_test_fe_v1 shape :", X_test_fe_v1.shape)

X_train_fe_v0 shape: (23814, 880)
X_train_fe_v1 shape: (23814, 892)
X_test_fe_v0 shape : (3982, 880)
X_test_fe_v1 shape : (3982, 892)


In [64]:
fe_v1_added_features = gene_summary_features + cell_summary_features

fe_v1_column_growth_audit = pd.DataFrame({
    "feature_set": [
        "FE_V0",
        "FE_V1"
    ],
    "train_columns_including_id": [
        X_train_fe_v0.shape[1],
        X_train_fe_v1.shape[1]
    ],
    "test_columns_including_id": [
        X_test_fe_v0.shape[1],
        X_test_fe_v1.shape[1]
    ],
    "model_features_excluding_id": [
        X_train_fe_v0.shape[1] - 1,
        X_train_fe_v1.shape[1] - 1
    ],
    "new_features_added_vs_previous": [
        0,
        len(fe_v1_added_features)
    ]
})

fe_v1_column_growth_audit

,feature_set,train_columns_including_id,test_columns_including_id,model_features_excluding_id,new_features_added_vs_previous
0,FE_V0,880,880,879,0
1,FE_V1,892,892,891,12


In [65]:
fe_v1_train_columns_without_id = [col for col in X_train_fe_v1.columns if col != ID_COL]
fe_v1_test_columns_without_id = [col for col in X_test_fe_v1.columns if col != ID_COL]

fe_v1_alignment_audit = {
    "train_rows": X_train_fe_v1.shape[0],
    "test_rows": X_test_fe_v1.shape[0],
    "train_columns": X_train_fe_v1.shape[1],
    "test_columns": X_test_fe_v1.shape[1],
    "train_test_columns_match": fe_v1_train_columns_without_id == fe_v1_test_columns_without_id,
    "train_sig_id_order_preserved": X_train_fe_v1[ID_COL].tolist() == train_features_clean[ID_COL].tolist(),
    "test_sig_id_order_preserved": X_test_fe_v1[ID_COL].tolist() == test_features_clean[ID_COL].tolist(),
    "fe_v0_columns_preserved": X_train_fe_v1[X_train_fe_v0.columns].columns.tolist() == X_train_fe_v0.columns.tolist(),
    "summary_feature_count_added": len(fe_v1_added_features)
}

fe_v1_alignment_audit

{'train_rows': 23814,
 'test_rows': 3982,
 'train_columns': 892,
 'test_columns': 892,
 'train_test_columns_match': True,
 'train_sig_id_order_preserved': True,
 'test_sig_id_order_preserved': True,
 'fe_v0_columns_preserved': True,
 'summary_feature_count_added': 12}

In [66]:
fe_v1_feature_cols = [col for col in X_train_fe_v1.columns if col != ID_COL]

def missing_infinite_audit(df, dataset_name, feature_cols):
    values = df[feature_cols].to_numpy(dtype=np.float64)
    
    return {
        "dataset": dataset_name,
        "rows": df.shape[0],
        "features_excluding_id": len(feature_cols),
        "missing_values": int(df[feature_cols].isna().sum().sum()),
        "positive_infinite_values": int(np.isposinf(values).sum()),
        "negative_infinite_values": int(np.isneginf(values).sum()),
        "all_values_finite": bool(np.isfinite(values).all())
    }


fe_v1_missing_infinite_audit = pd.DataFrame([
    missing_infinite_audit(X_train_fe_v1, "X_train_FE_V1", fe_v1_feature_cols),
    missing_infinite_audit(X_test_fe_v1, "X_test_FE_V1", fe_v1_feature_cols)
])

fe_v1_missing_infinite_audit

,dataset,rows,features_excluding_id,missing_values,positive_infinite_values,negative_infinite_values,all_values_finite
0,X_train_FE_V1,23814,891,0,0,0,True
1,X_test_FE_V1,3982,891,0,0,0,True


In [67]:
fe_v1_forbidden_train_found = sorted(
    list(set(X_train_fe_v1.columns).intersection(forbidden_input_columns))
)

fe_v1_forbidden_test_found = sorted(
    list(set(X_test_fe_v1.columns).intersection(forbidden_input_columns))
)

fe_v1_forbidden_column_audit = pd.DataFrame({
    "dataset": [
        "X_train_FE_V1",
        "X_test_FE_V1"
    ],
    "forbidden_columns_found_count": [
        len(fe_v1_forbidden_train_found),
        len(fe_v1_forbidden_test_found)
    ],
    "forbidden_columns_found": [
        fe_v1_forbidden_train_found,
        fe_v1_forbidden_test_found
    ],
    "passed": [
        len(fe_v1_forbidden_train_found) == 0,
        len(fe_v1_forbidden_test_found) == 0
    ]
})

fe_v1_forbidden_column_audit

,dataset,forbidden_columns_found_count,forbidden_columns_found,passed
0,X_train_FE_V1,0,[],True
1,X_test_FE_V1,0,[],True


In [68]:
fe_v1_summary_sanity_rows = []

for dataset_name, df in [
    ("train", X_train_fe_v1),
    ("test", X_test_fe_v1)
]:
    for block_name in ["gene", "cell"]:
        min_col = f"{block_name}_min"
        max_col = f"{block_name}_max"
        range_col = f"{block_name}_range"
        std_col = f"{block_name}_std"
        abs_mean_col = f"{block_name}_abs_mean"
        
        fe_v1_summary_sanity_rows.append({
            "dataset": dataset_name,
            "block": block_name,
            "range_non_negative": bool((df[range_col] >= 0).all()),
            "std_non_negative": bool((df[std_col] >= 0).all()),
            "max_greater_equal_min": bool((df[max_col] >= df[min_col]).all()),
            "abs_mean_non_negative": bool((df[abs_mean_col] >= 0).all())
        })

fe_v1_summary_sanity_audit = pd.DataFrame(fe_v1_summary_sanity_rows)

fe_v1_summary_sanity_audit

,dataset,block,range_non_negative,std_non_negative,max_greater_equal_min,abs_mean_non_negative
0,train,gene,True,True,True,True
1,train,cell,True,True,True,True
2,test,gene,True,True,True,True
3,test,cell,True,True,True,True


In [69]:
fe_v1_summary_distribution_rows = []

for feature in fe_v1_added_features:
    train_mean = X_train_fe_v1[feature].mean()
    test_mean = X_test_fe_v1[feature].mean()
    train_std = X_train_fe_v1[feature].std()
    test_std = X_test_fe_v1[feature].std()
    
    standardized_mean_diff = abs(train_mean - test_mean) / (train_std + 1e-8)
    
    fe_v1_summary_distribution_rows.append({
        "feature": feature,
        "train_mean": train_mean,
        "test_mean": test_mean,
        "train_std": train_std,
        "test_std": test_std,
        "standardized_mean_diff": standardized_mean_diff
    })

fe_v1_summary_distribution_audit = (
    pd.DataFrame(fe_v1_summary_distribution_rows)
    .sort_values("standardized_mean_diff", ascending=False)
    .reset_index(drop=True)
)

fe_v1_summary_distribution_audit

,feature,train_mean,test_mean,train_std,test_std,standardized_mean_diff
0,cell_range,3.507957,3.396253,1.741175,1.615830,0.064154
1,cell_std,0.697236,0.675322,0.384265,0.356320,0.057028
2,cell_min,-2.321465,-2.205862,2.257917,2.162083,0.051199
3,gene_max,5.144619,5.074354,1.855281,1.802375,0.037873
4,gene_range,10.356186,10.230064,3.379736,3.298702,0.037317
5,gene_min,-5.211566,-5.155712,1.836459,1.804463,0.030414
6,gene_std,1.032783,1.012225,0.678894,0.668992,0.030282
7,gene_abs_mean,0.760691,0.745159,0.532184,0.525620,0.029186
8,cell_mean,-0.432231,-0.394335,1.732085,1.726189,0.021879
9,cell_abs_mean,0.961613,0.939561,1.563949,1.558370,0.014101


In [70]:
assert fe_v1_alignment_audit["train_test_columns_match"], "FE_V1 train/test columns do not match"
assert fe_v1_alignment_audit["train_sig_id_order_preserved"], "FE_V1 train sig_id order changed"
assert fe_v1_alignment_audit["test_sig_id_order_preserved"], "FE_V1 test sig_id order changed"
assert fe_v1_missing_infinite_audit["all_values_finite"].all(), "FE_V1 contains missing or infinite values"
assert fe_v1_forbidden_column_audit["passed"].all(), "Forbidden columns found in FE_V1"

assert fe_v1_summary_sanity_audit[
    ["range_non_negative", "std_non_negative", "max_greater_equal_min", "abs_mean_non_negative"]
].all().all(), "FE_V1 summary sanity check failed"

print("Step 6 FE_V1 hard checks passed successfully.")

Step 6 FE_V1 hard checks passed successfully.


In [71]:
X_train_fe_v1.to_parquet(
    DATA_PROCESSED_DIR / "X_train_FE_V1.parquet",
    index=False
)

X_test_fe_v1.to_parquet(
    DATA_PROCESSED_DIR / "X_test_FE_V1.parquet",
    index=False
)

fe_v1_summary_feature_definition_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_6_fe_v1_summary_feature_definition_audit.csv",
    index=False
)

fe_v1_column_growth_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_6_fe_v1_column_growth_audit.csv",
    index=False
)

pd.DataFrame([fe_v1_alignment_audit]).to_csv(
    EXPERIMENT_LOG_DIR / "step_6_fe_v1_alignment_audit.csv",
    index=False
)

fe_v1_missing_infinite_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_6_fe_v1_missing_infinite_audit.csv",
    index=False
)

fe_v1_forbidden_column_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_6_fe_v1_forbidden_column_audit.csv",
    index=False
)

fe_v1_summary_sanity_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_6_fe_v1_summary_sanity_audit.csv",
    index=False
)

fe_v1_summary_distribution_audit.to_csv(
    EXPERIMENT_LOG_DIR / "step_6_fe_v1_summary_distribution_audit.csv",
    index=False
)

print("Step 6 FE_V1 matrices and audit files saved successfully.")

Step 6 FE_V1 matrices and audit files saved successfully.


In [72]:
fe_v1_manifest = {
    "feature_set": "FE_V1",
    "description": "FE_V0 plus deterministic row-wise gene and cell summary features.",
    "base_feature_set": "FE_V0",
    "id_column": ID_COL,
    "added_features": fe_v1_added_features,
    "gene_summary_features": gene_summary_features,
    "cell_summary_features": cell_summary_features,
    "added_feature_count": len(fe_v1_added_features),
    "train_shape": list(X_train_fe_v1.shape),
    "test_shape": list(X_test_fe_v1.shape),
    "forbidden_inputs_excluded": True,
    "scored_targets_excluded": True,
    "nonscored_targets_excluded": True,
    "drug_id_excluded": True,
    "learned_transformations_used": False,
    "saved_files": {
        "X_train_FE_V1": str(DATA_PROCESSED_DIR / "X_train_FE_V1.parquet"),
        "X_test_FE_V1": str(DATA_PROCESSED_DIR / "X_test_FE_V1.parquet"),
        "y_scored": str(DATA_PROCESSED_DIR / "y_scored.parquet")
    }
}

with open(DATA_PROCESSED_DIR / "FE_V1_manifest.json", "w", encoding="utf-8") as f:
    json.dump(fe_v1_manifest, f, indent=2)

fe_v1_manifest

{'feature_set': 'FE_V1',
 'description': 'FE_V0 plus deterministic row-wise gene and cell summary features.',
 'base_feature_set': 'FE_V0',
 'id_column': 'sig_id',
 'added_features': ['gene_mean',
  'gene_std',
  'gene_abs_mean',
  'gene_min',
  'gene_max',
  'gene_range',
  'cell_mean',
  'cell_std',
  'cell_abs_mean',
  'cell_min',
  'cell_max',
  'cell_range'],
 'gene_summary_features': ['gene_mean',
  'gene_std',
  'gene_abs_mean',
  'gene_min',
  'gene_max',
  'gene_range'],
 'cell_summary_features': ['cell_mean',
  'cell_std',
  'cell_abs_mean',
  'cell_min',
  'cell_max',
  'cell_range'],
 'added_feature_count': 12,
 'train_shape': [23814, 892],
 'test_shape': [3982, 892],
 'forbidden_inputs_excluded': True,
 'scored_targets_excluded': True,
 'nonscored_targets_excluded': True,
 'drug_id_excluded': True,
 'learned_transformations_used': False,
 'saved_files': {'X_train_FE_V1': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\moa-prediction-drug-response\\data

In [73]:
registry_path = DATA_PROCESSED_DIR / "feature_set_registry.json"

if registry_path.exists():
    with open(registry_path, "r", encoding="utf-8") as f:
        feature_set_registry = json.load(f)
else:
    feature_set_registry = {"available_feature_sets": {}, "next_planned_feature_sets": {}}

feature_set_registry["available_feature_sets"]["FE_V1"] = {
    "description": "FE_V0 plus deterministic row-wise gene/cell summary features.",
    "train_path": str(DATA_PROCESSED_DIR / "X_train_FE_V1.parquet"),
    "test_path": str(DATA_PROCESSED_DIR / "X_test_FE_V1.parquet"),
    "target_path": str(DATA_PROCESSED_DIR / "y_scored.parquet"),
    "manifest_path": str(DATA_PROCESSED_DIR / "FE_V1_manifest.json"),
    "train_shape": list(X_train_fe_v1.shape),
    "test_shape": list(X_test_fe_v1.shape),
    "target_shape": list(y_scored.shape),
    "feature_count_excluding_id": X_train_fe_v1.shape[1] - 1,
    "leakage_safe": True,
    "learned_transformations_used": False,
    "ready_for_model_training": True
}

feature_set_registry["next_planned_feature_sets"] = {
    "FE_V2": "FE_V1 + small audited gene-cell interaction features",
    "FE_V3": "Optional metadata interaction features"
}

with open(registry_path, "w", encoding="utf-8") as f:
    json.dump(feature_set_registry, f, indent=2)

feature_set_registry

{'available_feature_sets': {'FE_V0': {'description': 'Deterministic one-hot metadata + raw gene + raw cell features.',
   'train_path': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\moa-prediction-drug-response\\data\\processed\\X_train_FE_V0.parquet',
   'test_path': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\moa-prediction-drug-response\\data\\processed\\X_test_FE_V0.parquet',
   'target_path': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\moa-prediction-drug-response\\data\\processed\\y_scored.parquet',
   'manifest_path': 'C:\\Users\\USER\\Kawsar_Ahmmed\\ALL_Projects_Lab\\Machine_Learning\\moa-prediction-drug-response\\data\\processed\\FE_V0_manifest.json',
   'train_shape': [23814, 880],
   'test_shape': [3982, 880],
   'target_shape': [23814, 207],
   'feature_count_excluding_id': 879,
   'leakage_safe': True,
   'learned_transformations_used': False,
   'ready_for_model_training': True},
  'FE_V1': {'descrip

In [74]:
step_6_status = {
    "step": "Step 6 — Build FE_V1 Row-wise Gene/Cell Summary Features",
    "status": "completed",
    "fe_v1_saved": True,
    "summary_features_added": len(fe_v1_added_features),
    "train_test_columns_match": fe_v1_alignment_audit["train_test_columns_match"],
    "missing_infinite_check_passed": bool(fe_v1_missing_infinite_audit["all_values_finite"].all()),
    "summary_sanity_passed": bool(
        fe_v1_summary_sanity_audit[
            ["range_non_negative", "std_non_negative", "max_greater_equal_min", "abs_mean_non_negative"]
        ].all().all()
    ),
    "forbidden_column_check_passed": bool(fe_v1_forbidden_column_audit["passed"].all()),
    "learned_transformations_used": False,
    "ready_for_step_7": True,
    "next_step": "Step 7 — FE_V1 Audit and Handoff"
}

step_6_status

{'step': 'Step 6 — Build FE_V1 Row-wise Gene/Cell Summary Features',
 'status': 'completed',
 'fe_v1_saved': True,
 'summary_features_added': 12,
 'train_test_columns_match': True,
 'missing_infinite_check_passed': True,
 'summary_sanity_passed': True,
 'forbidden_column_check_passed': True,
 'learned_transformations_used': False,
 'ready_for_step_7': True,
 'next_step': 'Step 7 — FE_V1 Audit and Handoff'}